In [ ]:
import pandas as pd
import warnings
import sys
import glob

sys.path.append('/Users/irfan.hilman/ai-am')

from function import utils

warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.options.mode.chained_assignment = None

NEXT: BAGIAN EDIT DATA DIBUAT IPYNB TERPISAH!

Use Python 3.13.7

Cara penggunaan:
1. Update file Daily Price pada folder daily_stock_price
2. Update data Lapkeu secara Quarterly
3. RUN ALL

In [ ]:
ss_path = '/Users/irfan.hilman/finesia/stock_summary_ui'
tre_path = '/Users/irfan.hilman/finesia/tr_equity/TrEquity_from_2018.parquet'
is_path = '/Users/irfan.hilman/finesia/lapkeu_aggregated/Income_Statement_Aggregated.csv'
bs_path = '/Users/irfan.hilman/finesia/lapkeu_aggregated/Balance_Sheet_Aggregated.csv'
cf_path = '/Users/irfan.hilman/finesia/lapkeu_aggregated/Cash_Flow_Aggregated.csv'
cffr_path ='/Users/irfan.hilman/finesia/financial_ratio/Cash_Flow_Financial_Ratio.csv'
drop_stocks = ['excl','ihsg','MTSN','CNTB','CNTX']
max_quarter_ = 'Q12026'#'Q22025' # Untuk cap_quarter
recent_quarter_ = 'Q12026' # Mengikuti max_quarter_, namun format 'MMYYYY'
quarters = ['Q32026','Q22026','Q12026','Q42025','Q32025','Q22025','Q12025','Q42024'] # Selalu mulai dari Q42024, taruh di list[-1]
quarter_year_to_fill_ = 'Q12026' # Untuk fill data laporan saham yg belum lengkap
quarter_year_source_ = 'Q32025' # Untuk fill data laporan saham yg belum lengkap
quarter_num_path_ = '/Users/irfan.hilman/finesia/additional_data/quarter_num_update_Q22026.xlsx'

bs_cols_list = ['Total Aset Lancar','Total Aset Tidak Lancar','Total Aset','Total Liabilitas Jangka Pendek','Total Liabilitas Jangka Panjang','Total Liabilitas','Total Ekuitas','Total Kepentingan Non Pengendali','Saham Beredar']
is_cols_list = ['Total Pendapatan','Laba Kotor','Laba Usaha','Laba Sebelum Pajak','Laba Bersih Tahun Berjalan']
cf_cols_list = ['Total Arus Kas Dari Aktivitas Operasi','Total Arus Kas Dari Aktivitas Investasi','Total Arus Kas Dari Aktivitas Pendanaan']

is_annualizing_list = ['Laba Bersih Tahun Berjalan','Total Pendapatan','Laba Usaha']
cf_annualizing_list = ['Total Arus Kas Dari Aktivitas Operasi','Free cash flow']

ihsg_path_ = '/Users/irfan.hilman/finesia/additional_data/IHSG investingcom.xlsx'
sector_path_ = '/Users/irfan.hilman/finesia/additional_data/sektor.xlsx'

In [ ]:
filenames = glob.glob(ss_path + "/*.xlsx")
tr_equity = pd.read_parquet(tre_path)
tr_equity = utils.augment_stock_summary_ui(data=tr_equity,file_name=filenames)
tr_equity = utils.delete_stocks(data=tr_equity,drop_list=drop_stocks)
tr_equity = utils.delete_warrants(data=tr_equity)
tr_equity = utils.last_listed_shares(data=tr_equity)
tr_equity  = utils.delete_duplicates(data=tr_equity ,param_combined='Kode_Date',param1='Kode',param2='Date')
tr_equity= utils.fill_zero_prices(data=tr_equity)
tr_equity = utils.add_sektor(tr_equity_data=tr_equity,sector_path=sector_path_)
tr_equity = tr_equity.sort_values(by='Date')
tr_equity = tr_equity.reset_index(drop=True)

income_statement = pd.read_csv(is_path)
income_statement['Quarter_Year'] = income_statement['Quarter_Year'].str.replace(' ', '')
income_statement = utils.cap_quarter(data=income_statement,quarter_num_path=quarter_num_path_,max_quarter=max_quarter_)
balance_sheet = pd.read_csv(bs_path)
balance_sheet['Quarter_Year'] = balance_sheet['Quarter_Year'].str.replace(' ', '')
balance_sheet = utils.cap_quarter(data=balance_sheet,quarter_num_path=quarter_num_path_,max_quarter=max_quarter_)
cash_flow = pd.read_csv(cf_path)
cash_flow['Quarter_Year'] = cash_flow['Quarter_Year'].str.replace(' ', '')
cash_flow  = utils.cap_quarter(data=cash_flow ,quarter_num_path=quarter_num_path_,max_quarter=max_quarter_)
cash_flow_fin_ratio = pd.read_csv(cffr_path)
cash_flow_fin_ratio = cash_flow_fin_ratio.rename(columns={'index':'Quarter_Year'})
cash_flow_fin_ratio['Quarter_Year'] = cash_flow_fin_ratio['Quarter_Year'].str.replace(' ', '')
cash_flow_fin_ratio  = utils.cap_quarter(data=cash_flow_fin_ratio ,quarter_num_path=quarter_num_path_,max_quarter=max_quarter_)
cash_flow_fin_ratio = cash_flow_fin_ratio.rename(columns={'Free cash flow (Quarter)':'Free cash flow'})
income_statement = income_statement[income_statement['Quarter_Year']!='Unnamed: 36'] ## ada error di data TRUS # Delete jika file IPYNB edit data sudah terpisah!!
basic_cols = ['Kode','Quarter_Year']
bs_cols = basic_cols.copy()
bs_cols.extend(bs_cols_list)
is_cols = basic_cols.copy()
is_cols.extend(is_cols_list)
cf_cols = basic_cols.copy()
cf_cols.extend(cf_cols_list)
balance_sheet = balance_sheet[bs_cols]
income_statement = income_statement[is_cols]
cash_flow = cash_flow[cf_cols]
balance_sheet = utils.quarter_year(data=balance_sheet)
income_statement = utils.quarter_year(data=income_statement)
cash_flow = utils.quarter_year(data=cash_flow)
cash_flow = utils.add_financial_ratio(database_data=cash_flow,financial_ratio_data=cash_flow_fin_ratio,financial_ratio_param=['Free cash flow'])
cash_flow = utils.delete_duplicates(data=cash_flow,param_combined='Kode_Quarter_Year',param1='Kode',param2='Quarter_Year')
balance_sheet = utils.add_top_quarter(data=balance_sheet,quarter_list=quarters)
income_statement = utils.add_top_quarter(data=income_statement,quarter_list=quarters)
cash_flow = utils.add_top_quarter(data=cash_flow,quarter_list=quarters)
balance_sheet = utils.kode_quarter(data=balance_sheet)
income_statement = utils.kode_quarter(data=income_statement)
cash_flow = utils.kode_quarter(data=cash_flow)
income_statement = utils.create_quarter_number(data=income_statement,quarter_num_path=quarter_num_path_)
cash_flow = utils.create_quarter_number(data=cash_flow,quarter_num_path=quarter_num_path_)
balance_sheet = balance_sheet.reset_index(drop=True)
income_statement = income_statement.reset_index(drop=True)
cash_flow = cash_flow.reset_index(drop=True)
income_statement = utils.annualizing(data=income_statement,annualizing_list=income_statement.columns[4:-2],keep=is_annualizing_list)
cash_flow = utils.annualizing(data=cash_flow,annualizing_list=cash_flow.columns[4:-2],keep=cf_annualizing_list)

balance_sheet = utils.quarter_equalization(data=balance_sheet,quarter_num_path=quarter_num_path_)
income_statement = utils.quarter_equalization(data=income_statement,quarter_num_path=quarter_num_path_)
cash_flow = utils.quarter_equalization(data=cash_flow,quarter_num_path=quarter_num_path_)
balance_sheet = utils.ffill_recent_quarter(data=balance_sheet,quarter_year_to_fill=quarter_year_to_fill_,quarter_year_source=quarter_year_source_)
income_statement = utils.ffill_recent_quarter(data=income_statement,quarter_year_to_fill=quarter_year_to_fill_,quarter_year_source=quarter_year_source_)
cash_flow = utils.ffill_recent_quarter(data=cash_flow,quarter_year_to_fill=quarter_year_to_fill_,quarter_year_source=quarter_year_source_)
balance_sheet = utils.create_quarter_number(data=balance_sheet,quarter_num_path=quarter_num_path_)
income_statement = utils.create_quarter_number(data=income_statement,quarter_num_path=quarter_num_path_)
cash_flow = utils.create_quarter_number(data=cash_flow,quarter_num_path=quarter_num_path_)
balance_sheet = utils.kode_quarter(data=balance_sheet)
income_statement = utils.kode_quarter(data=income_statement)
cash_flow = utils.kode_quarter(data=cash_flow)
balance_sheet = utils.lag_2(data=balance_sheet,list_lag_params=balance_sheet.columns[4:-3],recent_qy=recent_quarter_)
income_statement = utils.lag_2(data=income_statement,list_lag_params=income_statement.columns[6:],recent_qy=recent_quarter_)
cash_flow = utils.lag_2(data=cash_flow,list_lag_params=cash_flow.columns[6:],recent_qy=recent_quarter_)
balance_sheet = utils.kode_quarter_year(data=balance_sheet)
income_statement = utils.kode_quarter_year(data=income_statement)
cash_flow = utils.kode_quarter_year(data=cash_flow)

lapkeu = income_statement.merge(balance_sheet.drop(['Kode','Quarter','Year','Quarter_Year','Kode_Quarter','Quarter Number'],axis=1), how='outer', on='Kode_Quarter_Year')
lapkeu = utils.delete_duplicates(data=lapkeu,param_combined='Kode_Quarter_Year',param1='Kode',param2='Quarter_Year')
lapkeu = utils.kode_quarter_year(data=lapkeu)
lapkeu = lapkeu.merge(cash_flow.drop(['Kode','Quarter','Year','Quarter_Year','Kode_Quarter','Quarter Number'],axis=1), how='outer', on='Kode_Quarter_Year')
lapkeu = utils.delete_duplicates(data=lapkeu,param_combined='Kode_Quarter_Year',param1='Kode',param2='Quarter_Year')
lapkeu = utils.date_from_to(data=lapkeu)
lapkeu = utils.complete_outstanding_shares(data=lapkeu)

lapkeu_all_date = utils.create_all_date(data=lapkeu)
lapkeu_all_date = utils.kode_date(data=lapkeu_all_date)
lapkeu_all_date = lapkeu_all_date.drop(columns=['Quarter Number','Kode_Quarter','Date From','Date To'])
tr_equity = utils.kode_date(data=tr_equity)

database = tr_equity.merge(lapkeu_all_date.drop(['Kode','Date'],axis=1), how='left', on='Kode_Date')
database = database.drop('Kode_Date',axis=1)
database = utils.age(data=database)
database = utils.add_IHSG(data=database, columns=['Date','IHSG Close Price'], path_=ihsg_path_)
database = utils.delete_duplicates(data=database,param_combined='Kode_Date',param1='Kode',param2='Date')
database['Market Cap'] = database['Close Price']*database['Listed Share']

# ADD PARAMS
database = utils.add_sales_to_market_cap(data=database)
database = utils.add_Frazzini_beta(data=database)
database = utils.delete_duplicates(data=database,param_combined='Kode_Date',param1='Kode',param2='Date')

# Export
database = database.drop(columns='Kode_Quarter_Year')
database = database.sort_values(by='Date')
cols_to_convert = database.columns.difference(['Kode', 'Date','Nama Perusahaan'])
database[cols_to_convert] = database[cols_to_convert].apply(pd.to_numeric, errors='coerce')
database = database.reset_index(drop=True)
database.to_parquet('/Users/irfan.hilman/ai-am/database_daily_update/Database.parquet',index=False)

All data in list_lag_params has been lagged 2: Index(['Total Aset Lancar', 'Total Aset Tidak Lancar', 'Total Aset',
       'Total Liabilitas Jangka Pendek', 'Total Liabilitas Jangka Panjang',
       'Total Liabilitas', 'Total Ekuitas',
       'Total Kepentingan Non Pengendali'],
      dtype='object')
All data in list_lag_params has been lagged 2: Index(['Total Pendapatan', 'Laba Kotor', 'Laba Usaha', 'Laba Sebelum Pajak',
       'Laba Bersih Tahun Berjalan', 'Laba Bersih Tahun Berjalan (quarterly)',
       'Total Pendapatan (quarterly)', 'Laba Usaha (quarterly)'],
      dtype='object')
All data in list_lag_params has been lagged 2: Index(['Total Arus Kas Dari Aktivitas Operasi',
       'Total Arus Kas Dari Aktivitas Investasi',
       'Total Arus Kas Dari Aktivitas Pendanaan', 'Free cash flow',
       'Total Arus Kas Dari Aktivitas Operasi (quarterly)',
       'Free cash flow (quarterly)'],
      dtype='object')
